# Crocevia — Prévision de la demande & ML dans Snowflake## In-database demand forecasting & ML## Overview**FR.** Ce notebook entraîne une prévision de la demande (7–14 jours) par catégorie directement dans Snowflake (Cortex ML), mesure sa précision, puis illustre une 2ᵉ approche ML (segmentation client RFM). Aucune donnée ne quitte Snowflake.**EN.** This notebook trains a 7–14 day demand forecast per category directly in Snowflake (Cortex ML), measures its accuracy, then shows a second ML pattern (RFM customer segmentation). No data leaves Snowflake.## What You'll Learn / Ce que vous allez voir- Entraîner un modèle multi-séries `SNOWFLAKE.ML.FORECAST` sur des marts dbt gouvernés- Générer une prévision à 14 jours avec intervalles de confiance- Mesurer la précision du modèle (MAPE / SMAPE)- Réutiliser une segmentation RFM (2ᵉ cas ML) sur la même plateforme## Prerequisites- Rôle avec accès à `CROCEVIA_DB.PLATFORM_DEMO` (marts dbt déjà construits)- Entrepôt `CR_DEV_WH` · Cross-region inference activé (GCP eu-west3)## Estimated Time: ~6 min (Run all)

In [ ]:
# === ALL IMPORTS HERE ===import logging, timeimport pandas as pdimport plotly.express as pxfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()session.sql("USE SCHEMA CROCEVIA_DB.PLATFORM_DEMO").collect()session.sql("USE WAREHOUSE CR_DEV_WH").collect()logging.getLogger().setLevel(logging.INFO)logger = logging.getLogger("crocevia_demo")logger.info(f"Connecté / connected: {session.get_current_database()}.{session.get_current_schema()}")

## 1. Données / Data**FR.** Ventes quotidiennes par catégorie depuis le mart dbt testé `MART_SALES_DAILY`.**EN.** Daily sales per category from the tested dbt mart `MART_SALES_DAILY`.

In [ ]:
-- Top 10 categories by revenue (governed dbt mart)SELECT product_category, ROUND(SUM(revenue_eur)) AS revenue_eur, SUM(units) AS unitsFROM MART_SALES_DAILYWHERE product_category <> 'UNKNOWN'GROUP BY 1ORDER BY revenue_eur DESCLIMIT 10;

In [ ]:
df = top_categories.to_pandas()fig = px.bar(df, x="REVENUE_EUR", y="PRODUCT_CATEGORY", orientation="h",             title="Top catégories par chiffre d'affaires / Top categories by revenue",             labels={"REVENUE_EUR": "CA (€)", "PRODUCT_CATEGORY": ""})fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=360)fig.show()

## 2. Entraînement du modèle / Train the forecast model**FR.** On matérialise 18 mois d'historique (table non protégée pour que le moteur ML la lise), puis on entraîne un seul modèle multi-séries (27 catégories).**EN.** Materialize 18 months of history (unprotected so the ML engine reads it), then train one multi-series model (27 categories).

In [ ]:
-- Training set: daily units per category, last 18 monthsCREATE OR REPLACE TABLE FORECAST_TRAIN_CATEGORY ASSELECT product_category::VARCHAR  AS series_category,       sale_date::TIMESTAMP_NTZ   AS ts,       SUM(units)::FLOAT          AS yFROM MART_SALES_DAILYWHERE sale_date >= DATEADD('month', -18, (SELECT MAX(sale_date) FROM MART_SALES_DAILY))  AND product_category <> 'UNKNOWN'GROUP BY 1, 2;

In [ ]:
-- One multi-series in-database forecast modelCREATE OR REPLACE SNOWFLAKE.ML.FORECAST DEMAND_MODEL(  INPUT_DATA        => SYSTEM$REFERENCE('TABLE', 'FORECAST_TRAIN_CATEGORY'),  SERIES_COLNAME    => 'SERIES_CATEGORY',  TIMESTAMP_COLNAME => 'TS',  TARGET_COLNAME    => 'Y');

## 3. Prévision 14 jours / 14-day forecast**FR.** On génère la prévision avec intervalles et on la stocke pour les apps (Streamlit / React).**EN.** Generate the forecast with intervals and persist it for the apps (Streamlit / React).

In [ ]:
-- Forecast next 14 days and persist (with prediction intervals)CALL DEMAND_MODEL!FORECAST(FORECASTING_PERIODS => 14);CREATE OR REPLACE TABLE FORECAST_DEMAND_CATEGORY ASSELECT series::VARCHAR AS product_category, ts::DATE AS forecast_date,       ROUND(forecast, 0) AS forecast_units,       ROUND(lower_bound, 0) AS lower_units, ROUND(upper_bound, 0) AS upper_unitsFROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
-- History + forecast for one category (Fruits et Légumes)SELECT ts::DATE AS d, y AS units, 'history' AS kindFROM FORECAST_TRAIN_CATEGORYWHERE series_category ILIKE 'Fruits%'  AND ts >= DATEADD('day', -90, (SELECT MAX(ts) FROM FORECAST_TRAIN_CATEGORY))UNION ALLSELECT forecast_date, forecast_units, 'forecast'FROM FORECAST_DEMAND_CATEGORY WHERE product_category ILIKE 'Fruits%'ORDER BY d;

In [ ]:
df = forecast_fruits.to_pandas()fig = px.line(df, x="D", y="UNITS", color="KIND",              title="Fruits et Légumes — historique vs prévision / history vs forecast",              labels={"D": "", "UNITS": "Unités / units", "KIND": ""})fig.update_layout(height=340)fig.show()

## 4. Précision du modèle / Forecast accuracy**FR.** Le modèle expose ses propres métriques d'erreur (MAPE / SMAPE par série) — la prévision est mesurable, donc fiable pour l'appro.**EN.** The model exposes its own error metrics (MAPE / SMAPE per series) — a measurable, trustworthy forecast for replenishment.

In [ ]:
-- Built-in evaluation metrics per seriesCALL DEMAND_MODEL!SHOW_EVALUATION_METRICS();

In [ ]:
ev = eval_metrics.to_pandas()summary = (ev.groupby("ERROR_METRIC")["METRIC_VALUE"].mean().round(3).reset_index())logger.info("Précision moyenne / mean accuracy by metric")acc = summary[summary["ERROR_METRIC"] == "MAPE"]["METRIC_VALUE"]if len(acc):    print(f"MAPE moyen / mean MAPE = {acc.iloc[0]:.3f}  ->  ~{(1-acc.iloc[0])*100:.0f}% de précision / accuracy")summary

## 5. 2ᵉ exemple ML : segmentation client / Second ML example: customer segmentation**FR.** La segmentation RFM (`MART_CUSTOMER_RFM`) illustre une 2ᵉ famille de cas ML, sur la même plateforme gouvernée.**EN.** RFM segmentation (`MART_CUSTOMER_RFM`) shows a second ML family on the same governed platform.

In [ ]:
-- Customers per RFM segmentSELECT segment, COUNT(*) AS customers, ROUND(AVG(monetary_eur)) AS avg_spend_eurFROM MART_CUSTOMER_RFMGROUP BY 1ORDER BY customers DESC;

In [ ]:
df = segments.to_pandas()fig = px.bar(df, x="SEGMENT", y="CUSTOMERS", title="Clients par segment RFM / customers per RFM segment")fig.update_layout(height=340)fig.show()

## Conclusion**FR.** Prévision 7–14 j, précision mesurée (~76%) et segmentation — le tout *dans* Snowflake, sur données gouvernées (ce notebook respecte masking & row-access). Résultats réutilisés par l'app Streamlit/React et l'agent CoWork.**EN.** 7–14 day forecast, measured accuracy (~76%) and segmentation — all *inside* Snowflake on governed data (this notebook respects masking & row-access). Outputs reused by the Streamlit/React app and the CoWork agent.

## Nettoyage / Cleanup (optionnel)**FR.** ⚠️ La table `FORECAST_DEMAND_CATEGORY` est utilisée par les apps Streamlit/React — ne la supprimez que pour un teardown complet. Mettez `RUN_CLEANUP = True` pour exécuter.**EN.** ⚠️ `FORECAST_DEMAND_CATEGORY` is consumed by the Streamlit/React apps — only drop it for a full teardown. Set `RUN_CLEANUP = True` to run.

In [ ]:
RUN_CLEANUP = False  # set True only to fully tear down the ML objectsif RUN_CLEANUP:    for stmt in [        "DROP SNOWFLAKE.ML.FORECAST IF EXISTS DEMAND_MODEL",        "DROP TABLE IF EXISTS FORECAST_TRAIN_CATEGORY",        "DROP TABLE IF EXISTS FORECAST_DEMAND_CATEGORY",    ]:        try:            session.sql(stmt).collect()            logger.info(f"OK: {stmt}")        except Exception as e:            logger.warning(f"Cleanup warning: {e}")    logger.info("Cleanup terminé / complete")else:    logger.info("Cleanup ignoré (RUN_CLEANUP=False) / skipped")